# Assignment 1 — training pipeline

You're using a notebook for this stage. If you'd rather use a script,
write `train.py` instead and delete this file — `main.py` refuses to run
if it finds both `train.py` and `train.ipynb`, so exactly one of them must
exist.

Replace this cell and everything below it with your own code. When run via
`uv run python main.py train`, this notebook must, start to finish, with
no manual steps in between:

- Load your committed dataset.
- Apply the feature engineering, preprocessing, and train/val/test split
  consistent with the business framing you wrote in `REPORT.md`.
- Train the same model — linear regression for a regression target,
  logistic regression for a binary one — three ways, on the same split:
  scikit-learn, a from-scratch PyTorch loop (Session 5's manual approach:
  raw tensors, autograd, a manual gradient-step update), and the standard
  `torch.nn.Module` + `torch.optim` workflow.
- Evaluate all three against each other and against a naive baseline.
- Save whatever `app.py`/`app.ipynb` needs to build its dashboard without
  retraining anything.

How you structure the code beyond that — cell layout, function
boundaries, naming — is your call to make and be able to defend.

**Data Exploring**

In [1]:
import pandas as pd

df = pd.read_csv("data/aep_raw_sbd.csv", parse_dates=["Date"])
print("Raw rows:", len(df))


df = df[~df["Place"].isin(["DD", "NS"])]
df = df.drop_duplicates(["LifterID", "Date", "MeetName", "TotalKg"])
same_day = df[df.duplicated(["LifterID", "Date"], keep=False)]["LifterID"]
valid = df.dropna(subset=["TotalKg"]).sort_values(["LifterID", "Date"])
rel_change = valid.groupby("LifterID")["TotalKg"].pct_change()
big_drop = valid.loc[rel_change < -0.25, "LifterID"]
suspicious = set(same_day) | set(big_drop)
df = df[~df["LifterID"].isin(suspicious)]

df = df.sort_values(["Date", "LifterID"]).reset_index(drop=True)
print("Suspicious lifter IDs removed:", len(suspicious))
print("Clean rows:", len(df), "| lifters:", df["LifterID"].nunique())

Raw rows: 14102
Suspicious lifter IDs removed: 12
Clean rows: 14004 | lifters: 5887


In [2]:
import numpy as np

df = df.sort_values(["LifterID", "Date"]).reset_index(drop=True)
df["is_valid"] = df["TotalKg"].notna()          

v = df[df["is_valid"]].copy()
ATTEMPTS = [f"{lift}{i}Kg" for lift in ["Squat", "Bench", "Deadlift"] for i in (1, 2, 3)]
v["misses"] = (v[ATTEMPTS] < 0).sum(axis=1)                    # failed attempts at that meet
g = v.groupby("LifterID")
v["prev_misses"] = g["misses"].shift(1)                         # ...at the PREVIOUS valid meet
v["prev_total"] = g["TotalKg"].shift(1)                         
v["prev2_total"] = g["TotalKg"].shift(2)                        
v["best_prev_total"] = g["TotalKg"].transform(lambda s: s.shift(1).cummax())
v["prev_bw"] = g["BodyweightKg"].shift(1)                       
v["prev_date"] = g["Date"].shift(1)
v["first_date"] = g["Date"].transform("min")
v["n_prev_valid"] = g.cumcount()                                
df = df.merge(v[["prev_total", "prev2_total", "best_prev_total", "prev_bw",
                 "prev_date", "first_date", "n_prev_valid","prev_misses"]],
              left_index=True, right_index=True, how="left")

df["n_prev_dq"] = (~df["is_valid"]).astype(int).groupby(df["LifterID"]).cumsum() - (~df["is_valid"]).astype(int)

df["days_since_last"] = (df["Date"] - df["prev_date"]).dt.days
df["years_competing"] = (df["Date"] - df["first_date"]).dt.days / 365.25
df["trend"] = df["prev_total"] - df["prev2_total"]          
df["has_trend"] = df["trend"].notna().astype(int)
df["trend"] = df["trend"].fillna(0)
df["gap_to_best"] = df["best_prev_total"] - df["prev_total"] 
df["class_limit"] = df["WeightClassKg"].str.rstrip("+").astype(float)
df["is_plus_class"] = df["WeightClassKg"].str.endswith("+").astype(int)
df["class_vs_prev_bw"] = df["class_limit"] - df["prev_bw"]
df["male"] = (df["Sex"] == "M").astype(int)
df["is_national"] = df["MeetName"].str.contains(r"Espa[ñn]a|Nacional", case=False).astype(int)

model_df = df[df["is_valid"] & df["prev_total"].notna()].copy()
print("Rows we can predict:", len(model_df))

Rows we can predict: 7800


In [3]:
assert (model_df["prev_date"] < model_df["Date"]).all(), "previous meet is not earlier!"
assert (model_df["best_prev_total"] >= model_df["prev_total"]).all()
first_valid = df[df["is_valid"]].groupby("LifterID").head(1)
assert first_valid["prev_total"].isna().all(), "a first meet has a 'previous' total!"

lid = model_df["LifterID"].value_counts().index[0]
print(df[df["LifterID"] == lid][["Date", "TotalKg", "prev_total", "best_prev_total",
                                 "n_prev_valid", "n_prev_dq", "days_since_last"]].to_string())
print("All leakage checks passed")

           Date  TotalKg  prev_total  best_prev_total  n_prev_valid  n_prev_dq  days_since_last
5521 2016-02-27    522.5         NaN              NaN           0.0          0              NaN
5522 2017-02-11    595.0       522.5            522.5           1.0          0            350.0
5523 2017-04-01    577.5       595.0            595.0           2.0          0             49.0
5524 2017-11-25    592.5       577.5            595.0           3.0          0            238.0
5525 2018-03-09    613.0       592.5            595.0           4.0          0            104.0
5526 2018-06-30    636.0       613.0            613.0           5.0          0            113.0
5527 2019-04-05    661.5       636.0            636.0           6.0          0            279.0
5528 2021-03-19    667.5       661.5            661.5           7.0          0            714.0
5529 2021-06-04    698.0       667.5            667.5           8.0          0             77.0
5530 2022-04-01    707.5       698.0    

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

FEATURES = ["prev_total", "gap_to_best", "trend", "has_trend",
            "n_prev_valid", "n_prev_dq", "days_since_last", "years_competing",
            "Age", "male", "class_limit", "is_plus_class", "class_vs_prev_bw", "is_national","prev_misses"]
TARGET = "TotalKg"

train = model_df[model_df["Date"] < "2025-01-01"]
val = model_df[(model_df["Date"] >= "2025-01-01") & (model_df["Date"] < "2025-07-01")]
test = model_df[model_df["Date"] >= "2025-07-01"]
print(f"Train: {len(train)} | Validation: {len(val)} | Test: {len(test)}")

groups = [
    ("Previous total only", ["prev_total"]),
    ("+ history shape", ["gap_to_best", "trend", "has_trend"]),
    ("+ experience & timing", ["n_prev_valid", "years_competing", "n_prev_dq", "days_since_last"]),
    ("+ lifter (age, sex)", ["Age", "male"]),
    ("+ registration (weight class, national)", ["class_limit", "is_plus_class", "class_vs_prev_bw", "is_national"]),
    ("+ previous missed attempts", ["prev_misses"]),
]
cols, rows = [], []
for name, group in groups:
    cols = cols + group
    med = train[cols].median()
    m = LinearRegression().fit(train[cols].fillna(med), train[TARGET])
    mae = np.abs(m.predict(val[cols].fillna(med)) - val[TARGET]).mean()
    rows.append({"features": name, "val_MAE_kg": round(float(mae), 1)})
print(pd.DataFrame(rows).to_string(index=False))

medians = train[FEATURES].median()         
means = train[FEATURES].fillna(medians).mean()
stds = train[FEATURES].fillna(medians).std()

def prepare(part):
    X = (part[FEATURES].fillna(medians) - means) / stds
    return X.to_numpy(dtype="float32"), part[TARGET].to_numpy(dtype="float32")

X_train, y_train = prepare(train)
X_val, y_val = prepare(val)
X_test, y_test = prepare(test)

def evaluate(name, y_true, y_pred):
    err = y_pred - y_true                    
    return {"model": name,
            "MAE_kg": round(float(np.mean(np.abs(err))), 1),
            "R2": round(float(r2_score(y_true, y_pred)), 3),
            "mean_error_kg": round(float(err.mean()), 1),
            "overpredict_%": round(float(100 * (err > 0).mean()), 1)}

results = [evaluate("Baseline (same as last meet)", y_val, val["prev_total"].to_numpy())]
sk_model = LinearRegression().fit(X_train, y_train)
results.append(evaluate("scikit-learn", y_val, sk_model.predict(X_val)))

print(pd.DataFrame(results).to_string(index=False))

Train: 4923 | Validation: 1230 | Test: 1647
                               features  val_MAE_kg
                    Previous total only        23.2
                        + history shape        22.5
                  + experience & timing        19.6
                    + lifter (age, sex)        18.7
+ registration (weight class, national)        17.6
             + previous missed attempts        17.5
                       model  MAE_kg    R2  mean_error_kg  overpredict_%
Baseline (same as last meet)    29.9 0.911          -25.5           15.5
                scikit-learn    17.5 0.968           -3.3           45.4


In [5]:
import torch

torch.manual_seed(42)
Xt_train, yt_train = torch.tensor(X_train), torch.tensor(y_train).unsqueeze(1)
Xt_val = torch.tensor(X_val)

n_features = Xt_train.shape[1]
w = torch.zeros(n_features, 1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)
LR_MANUAL, EPOCHS_MANUAL = 0.1, 3000

for epoch in range(EPOCHS_MANUAL):
    pred = Xt_train @ w + b
    loss = ((pred - yt_train) ** 2).mean()   
    loss.backward()                             
    with torch.no_grad():                        
        w -= LR_MANUAL * w.grad
        b -= LR_MANUAL * b.grad
        w.grad.zero_()
        b.grad.zero_()
    if epoch % 500 == 0 or epoch == EPOCHS_MANUAL - 1:
        print(f"manual  epoch {epoch:4d}  train MSE {loss.item():10.1f}")

with torch.no_grad():
    manual_val_pred = (Xt_val @ w + b).squeeze(1).numpy()
results.append(evaluate("PyTorch manual loop", y_val, manual_val_pred))


class LinearRegressionModel(torch.nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.linear = torch.nn.Linear(n_features, 1)

    def forward(self, x):
        return self.linear(x)

nn_model = LinearRegressionModel(n_features)
loss_fn = torch.nn.MSELoss()
optimizer = torch.optim.SGD(nn_model.parameters(), lr=0.1)
EPOCHS_NN = 3000

for epoch in range(EPOCHS_NN):
    optimizer.zero_grad()
    loss = loss_fn(nn_model(Xt_train), yt_train)
    loss.backward()
    optimizer.step()
    if epoch % 500 == 0 or epoch == EPOCHS_NN - 1:
        print(f"nn.Module epoch {epoch:4d}  train MSE {loss.item():10.1f}")

nn_model.eval()
with torch.no_grad():
    nn_val_pred = nn_model(Xt_val).squeeze(1).numpy()
results.append(evaluate("PyTorch nn.Module", y_val, nn_val_pred))
print(pd.DataFrame(results).to_string(index=False))

coefs = pd.DataFrame({
    "scikit-learn": sk_model.coef_,
    "manual loop": w.detach().squeeze(1).numpy(),
    "nn.Module": nn_model.linear.weight.detach().squeeze(0).numpy(),
}, index=FEATURES)
coefs.loc["(intercept)"] = [sk_model.intercept_, b.item(), nn_model.linear.bias.item()]
print(coefs.round(2).to_string())
print("Max prediction difference vs scikit-learn (kg):",
      round(float(np.abs(manual_val_pred - sk_model.predict(X_val)).max()), 3), "(manual),",
      round(float(np.abs(nn_val_pred - sk_model.predict(X_val)).max()), 3), "(nn.Module)")

manual  epoch    0  train MSE   276432.4
manual  epoch  500  train MSE      527.0
manual  epoch 1000  train MSE      527.0
manual  epoch 1500  train MSE      527.0
manual  epoch 2000  train MSE      527.0
manual  epoch 2500  train MSE      527.0
manual  epoch 2999  train MSE      527.0
nn.Module epoch    0  train MSE   276391.6
nn.Module epoch  500  train MSE      527.0
nn.Module epoch 1000  train MSE      527.0
nn.Module epoch 1500  train MSE      527.0
nn.Module epoch 2000  train MSE      527.0
nn.Module epoch 2500  train MSE      527.0
nn.Module epoch 2999  train MSE      527.0
                       model  MAE_kg    R2  mean_error_kg  overpredict_%
Baseline (same as last meet)    29.9 0.911          -25.5           15.5
                scikit-learn    17.5 0.968           -3.3           45.4
         PyTorch manual loop    17.5 0.968           -3.3           45.4
           PyTorch nn.Module    17.5 0.968           -3.3           45.4
                  scikit-learn  manual loop  nn

In [6]:
import json
from pathlib import Path
import joblib

ART = Path("artifacts")
ART.mkdir(exist_ok=True)

Xt_test = torch.tensor(X_test)
with torch.no_grad():
    test_preds = {
        "baseline": test["prev_total"].to_numpy(),
        "scikit-learn": sk_model.predict(X_test),
        "PyTorch manual loop": (Xt_test @ w + b).squeeze(1).numpy(),
        "PyTorch nn.Module": nn_model(Xt_test).squeeze(1).numpy(),
    }
test_results = pd.DataFrame([evaluate(name, y_test, p) for name, p in test_preds.items()])
print("TEST SET RESULTS")
print(test_results.to_string(index=False))
joblib.dump(sk_model, ART / "sklearn_model.joblib")
torch.save({"w": w.detach(), "b": b.detach()}, ART / "manual_model.pt")
torch.save(nn_model.state_dict(), ART / "nn_model.pt")

with open(ART / "preprocessing.json", "w") as f:
    json.dump({"features": FEATURES, "medians": medians.to_dict(),
               "means": means.to_dict(), "stds": stds.to_dict()}, f, indent=2)

pred_table = test[["Date", "Sex", "WeightClassKg"] + FEATURES + [TARGET]].copy()
pred_table = pred_table.rename(columns={TARGET: "actual"})
for name, p in test_preds.items():
    pred_table[name] = p
pred_table.to_csv(ART / "test_predictions.csv", index=False)

model_df[FEATURES + [TARGET]].to_csv(ART / "feature_data.csv", index=False)
test_results.to_csv(ART / "test_metrics.csv", index=False)
coefs.to_csv(ART / "coefficients.csv")
print("Saved:", sorted(p.name for p in ART.iterdir()))

TEST SET RESULTS
              model  MAE_kg    R2  mean_error_kg  overpredict_%
           baseline    22.3 0.956          -15.5           22.5
       scikit-learn    16.2 0.977            1.5           52.8
PyTorch manual loop    16.2 0.977            1.5           52.8
  PyTorch nn.Module    16.2 0.977            1.5           52.8
Saved: ['coefficients.csv', 'feature_data.csv', 'manual_model.pt', 'nn_model.pt', 'preprocessing.json', 'sklearn_model.joblib', 'test_metrics.csv', 'test_predictions.csv']
